# Discovery — `qasmbench`, Silver table 6 (`conditional_correction`)

**Owner of this notebook:** the QASMBench *conditional corrections* table, Silver scope.

## What this notebook is

Read-only exploration of the `qasmbench` Bronze object, aimed at one question:
*what exactly goes into `silver/qasmbench/conditional_correction.parquet`, and how
do I know it is right?*

It reads bytes from MinIO, parses them **in memory**, and prints. It writes
nothing — not to disk, not to the bucket. The actual Silver build lives in
`src/quantum_lake_student/stages/`, run by a script; notebooks must stay out of
the pipeline (`faq.md`: "production stages and checks must be importable and run
by a documented command"; the rubric caps the grade at 60 if execution depends on
manually ordered notebook cells).

## The contract I have to satisfy

`silver-tables.md` §6 — path `silver/qasmbench/conditional_correction.parquet`,
**one row = one recovery operation controlled by a measured syndrome**:

| Column | Type | Meaning |
| --- | --- | --- |
| `source_record_id` | string | stable link to the correction statement |
| `circuit_id` | string | link to the Silver circuit (table 4) |
| `condition_register` | string | classical register used by the condition |
| `condition_value` | int64 | integer value that activates the operation |
| `gate` | string | correction gate name |
| `target_qubit` | string | qubit acted on by the correction |

Plus the rules that apply to every Silver table: faithful to the source, no
cross-source joins, invalid records excluded and written to
`results/part1/data_issues.parquet` with the observed value and a reason, and a
second run on unchanged input producing identical rows and identical ids.

## 0. Setup

`Settings.from_environment()` picks up the container env from `compose.yaml`
(`S3_ENDPOINT=http://minio:9000`, `S3_BUCKET=quantum-lake`). Nothing here is
extracted to the local filesystem.

In [1]:
import hashlib
import io
import re
import zipfile

import pandas as pd
import pyarrow as pa

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 60)

settings = Settings.from_environment()
client = minio_client(settings)

BRONZE_OBJECT = "bronze/source=qasmbench/qasmbench-qec.zip"
SOURCE_NAME = "qasmbench"


def fetch(key: str) -> bytes:
    response = client.get_object(settings.s3_bucket, key)
    try:
        return response.read()
    finally:
        response.close()
        response.release_conn()


BRONZE_BYTES = fetch(BRONZE_OBJECT)
INPUT_SHA256 = hashlib.sha256(BRONZE_BYTES).hexdigest()

print(f"object      : {settings.s3_bucket}/{BRONZE_OBJECT}")
print(f"bytes       : {len(BRONZE_BYTES):,}")
print(f"input_sha256: {INPUT_SHA256}")

object      : quantum-lake/bronze/source=qasmbench/qasmbench-qec.zip
bytes       : 144,172
input_sha256: 60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e4f85722a


## 1. Archive inventory, without extracting

Two things matter here.

**Safe member names.** `brief.md` is explicit that *"a missing required companion
file or unsafe archive member must stop the run"* — that is a hard failure, not a
`data_issues` row. An unsafe name is one that escapes the extraction root
(absolute, or containing `..`). We never extract, but the Silver stage must still
refuse to run on a hostile archive, so the check belongs in the parser too.

**What is and is not a circuit.** The archive carries `LICENSE`, `NOTICE`,
`README.md`, per-benchmark `README.md`, `.png` renders, and `qelib1.inc`.
`qelib1.inc` is the OpenQASM standard gate library — it is *included by* the
circuits, it is not a circuit. Only `*.qasm` under `small/<benchmark>/` is input.

In [2]:
archive = zipfile.ZipFile(io.BytesIO(BRONZE_BYTES))


def is_unsafe(member: str) -> bool:
    return member.startswith("/") or ".." in member.split("/")


inventory = pd.DataFrame(
    [
        {
            "archive_member": info.filename,
            "uncompressed_bytes": info.file_size,
            "unsafe_name": is_unsafe(info.filename),
            "is_circuit": info.filename.endswith(".qasm"),
        }
        for info in archive.infolist()
    ]
)

assert not inventory["unsafe_name"].any(), "unsafe archive member -> stop the run"
print(f"members: {len(inventory)}   circuits: {int(inventory.is_circuit.sum())}")
inventory

members: 19   circuits: 6


,archive_member,uncompressed_bytes,unsafe_name,is_circuit
0,LICENSE,2066,False,False
1,NOTICE,1285,False,False
2,README.md,27140,False,False
3,qelib1.inc,4090,False,False
4,small/error_correctiond3_n5/README.md,571,False,False
5,small/error_correctiond3_n5/error_correctiond3_n5.png,62789,False,False
6,small/error_correctiond3_n5/error_correctiond3_n5.qasm,1517,False,True
7,small/error_correctiond3_n5/error_correctiond3_n5_transp...,3304,False,True
8,small/error_correctiond3_n5/res_error_correctiond3_n5.png,16042,False,False
9,small/qec_en_n5/README.md,514,False,False


## 2. The circuit members

Three benchmark directories, each with a `source` and a `_transpiled` variant, so
**six circuits**. Both variants are real circuits and both get a row in Silver
table 4 — `silver-tables.md` §4 has an explicit `variant` column ("source or
transpiled variant"). That matters for me: a conditional correction belongs to a
*circuit variant*, not to a benchmark.

In [3]:
CIRCUIT_MEMBERS = sorted(n for n in archive.namelist() if n.endswith(".qasm"))

members = pd.DataFrame(
    [
        {
            "archive_member": member,
            "benchmark_name": member.split("/")[1],
            "variant": "transpiled" if member.endswith("_transpiled.qasm") else "source",
            "bytes": len(archive.read(member)),
        }
        for member in CIRCUIT_MEMBERS
    ]
)
members

,archive_member,benchmark_name,variant,bytes
0,small/error_correctiond3_n5/error_correctiond3_n5.qasm,error_correctiond3_n5,source,1517
1,small/error_correctiond3_n5/error_correctiond3_n5_transp...,error_correctiond3_n5,transpiled,3304
2,small/qec_en_n5/qec_en_n5.qasm,qec_en_n5,source,530
3,small/qec_en_n5/qec_en_n5_transpiled.qasm,qec_en_n5,transpiled,778
4,small/qec_sm_n5/qec_sm_n5.qasm,qec_sm_n5,source,377
5,small/qec_sm_n5/qec_sm_n5_transpiled.qasm,qec_sm_n5,transpiled,341


## 3. Read the one circuit that actually has corrections

`data-sources.md` calls `qec_sm_n5.qasm` *"the clearest parity-check example"*.
It is a three-qubit repetition code: three data qubits `q[3]`, two ancillas
`a[2]`, a two-bit syndrome register `syn[2]`, and three conditional `x`
corrections.

Read both variants side by side — they are the same circuit, written twice.

In [4]:
for member in [m for m in CIRCUIT_MEMBERS if "qec_sm_n5" in m]:
    print("=" * 78)
    print(member)
    print("=" * 78)
    print(archive.read(member).decode())

small/qec_sm_n5/qec_sm_n5.qasm
// Repetition code syndrome measurement
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
qreg a[2];
creg c[3];
creg syn[2];
gate syndrome d1,d2,d3,a1,a2 
{ 
  cx d1,a1; cx d2,a1; 
  cx d2,a2; cx d3,a2; 
}
x q[0]; // error
barrier q;
syndrome q[0],q[1],q[2],a[0],a[1];
measure a -> syn;
if(syn==1) x q[0];
if(syn==2) x q[2];
if(syn==3) x q[1];
measure q -> c;

small/qec_sm_n5/qec_sm_n5_transpiled.qasm
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
qreg a[2];
creg c[3];
creg syn[2];
x q[0];
barrier q[0],q[1],q[2];
cx q[0],a[0];
cx q[1],a[0];
cx q[1],a[1];
cx q[2],a[1];
measure a[0] -> syn[0];
measure a[1] -> syn[1];
if(syn==1) x q[0];
measure q[0] -> c[0];
if(syn==2) x q[2];
if(syn==3) x q[1];
measure q[1] -> c[1];
measure q[2] -> c[2];



### What the last four lines say

```text
measure a -> syn;      # the two ancillas become the two-bit syndrome
if(syn==1) x q[0];     # syndrome 01 -> undo an X error on data qubit 0
if(syn==2) x q[2];     # syndrome 10 -> undo an X error on data qubit 2
if(syn==3) x q[1];     # syndrome 11 -> undo an X error on data qubit 1
```

Each `if` line is exactly one row of my table. `condition_register` = `syn`,
`condition_value` = the integer, `gate` = `x`, `target_qubit` = the qubit.

The transpiled variant has the same three `if` lines, but the compiler has
**reordered them** relative to the final `measure q -> c` statements. Statement
position differs between variants; the corrections themselves do not. Keep that in
mind when choosing an id.

## 4. Does anything else in the archive have a condition?

Before writing a parser, check the whole archive with a crude text scan. If the
crude scan and the real parser disagree later, one of them is wrong — that
disagreement is the check.

In [5]:
scan = []
for info in archive.infolist():
    if info.filename.endswith(".png"):
        continue
    body = archive.read(info.filename).decode("utf-8", "replace")
    hits = [
        (line_no, line.strip())
        for line_no, line in enumerate(body.splitlines(), start=1)
        if re.search(r"\bif\s*\(", line)
    ]
    for line_no, line in hits:
        scan.append({"archive_member": info.filename, "line": line_no, "text": line})

CRUDE_CONDITION_COUNT = len(scan)
print(f"lines containing `if(` anywhere in the archive: {CRUDE_CONDITION_COUNT}")
pd.DataFrame(scan)

lines containing `if(` anywhere in the archive: 6


,archive_member,line,text
0,small/qec_sm_n5/qec_sm_n5.qasm,17,if(syn==1) x q[0];
1,small/qec_sm_n5/qec_sm_n5.qasm,18,if(syn==2) x q[2];
2,small/qec_sm_n5/qec_sm_n5.qasm,19,if(syn==3) x q[1];
3,small/qec_sm_n5/qec_sm_n5_transpiled.qasm,15,if(syn==1) x q[0];
4,small/qec_sm_n5/qec_sm_n5_transpiled.qasm,17,if(syn==2) x q[2];
5,small/qec_sm_n5/qec_sm_n5_transpiled.qasm,18,if(syn==3) x q[1];


Six. All in `qec_sm_n5`, three per variant. So the finished Silver table has
**six rows**, and the other four circuits contribute zero — an empty contribution
is a finding to record, not a bug to chase.

Note also that `qelib1.inc` contains no conditions: it is gate *definitions*.

## 5. Three parsing traps

`data-sources.md` and `quantum-data-primer.md` warn about these. Two of the three
bite my table directly.

**(a) A gate definition body is not executed code.**

```text
gate syndrome d1,d2,d3,a1,a2
{
  cx d1,a1; cx d2,a1;
  cx d2,a2; cx d3,a2;
}
```

Those four `cx` lines are a *definition*. The single line
`syndrome q[0],q[1],q[2],a[0],a[1];` is the execution. A naive split on `;` counts
the body as four extra statements, which shifts every statement index after it —
and my `source_record_id` is built from a statement index. So the tokenizer must
drop definition bodies.

**(b) Register-local indexing.** `q[0]` and `a[0]` are different qubits.
`target_qubit` therefore has to keep the register name; storing a bare integer
would be wrong.

**(c) An operation can address a whole register.** `measure a -> syn;` is two
executed measurements, not one. This trap is table 4's problem, not mine — but
the same shape could appear as `if(syn==1) x q;`, which *would* be mine. It does
not occur in this release; the parser still has to have an answer for it.

## 6. A minimal OpenQASM 2.0 statement tokenizer

Not a full parser — enough structure to find declarations and conditions
reliably, and to give every statement a repeatable position.

Rules: strip `//` comments; replace `gate ... { ... }` definitions with
equivalent newlines (so line numbers survive); split the rest on `;`; collapse
whitespace. Returns `(statement_index, line_number, text)`.

In [6]:
COMMENT = re.compile(r"//[^\n]*")
GATE_DEFINITION = re.compile(r"\bgate\b[^{};]*\{[^}]*\}", re.S)


def statements(text: str) -> list[tuple[int, int, str]]:
    """Executable statements as (statement_index, line_number, text).

    Comments are removed and gate-definition bodies are excluded, so the index
    counts executed statements only. Line numbers refer to the original file.
    """
    body = COMMENT.sub("", text)
    body = GATE_DEFINITION.sub(lambda m: "\n" * m.group(0).count("\n"), body)

    found: list[tuple[int, int, str]] = []
    index, line, start_line, buffer = 0, 1, None, []
    for character in body:
        if character == ";":
            statement = " ".join("".join(buffer).split())
            if statement:
                found.append((index, start_line, statement))
                index += 1
            buffer, start_line = [], None
        else:
            if start_line is None and not character.isspace():
                start_line = line
            buffer.append(character)
        if character == "\n":
            line += 1
    return found


for index, line_no, text in statements(archive.read("small/qec_sm_n5/qec_sm_n5.qasm").decode()):
    print(f"  stmt={index:<3} line={line_no:<3} {text}")

  stmt=0   line=2   OPENQASM 2.0
  stmt=1   line=3   include "qelib1.inc"
  stmt=2   line=4   qreg q[3]
  stmt=3   line=5   qreg a[2]
  stmt=4   line=6   creg c[3]
  stmt=5   line=7   creg syn[2]
  stmt=6   line=13  x q[0]
  stmt=7   line=14  barrier q
  stmt=8   line=15  syndrome q[0],q[1],q[2],a[0],a[1]
  stmt=9   line=16  measure a -> syn
  stmt=10  line=17  if(syn==1) x q[0]
  stmt=11  line=18  if(syn==2) x q[2]
  stmt=12  line=19  if(syn==3) x q[1]
  stmt=13  line=20  measure q -> c


Fourteen executed statements. The four `cx` lines inside `gate syndrome` are
absent, and `syndrome q[0],...` at `stmt=8` is present as a single invocation —
trap (a) handled. Line numbers still point at the real file, which is what makes a
finding checkable by hand.

## 7. Extracting the corrections

OpenQASM 2.0 allows exactly one conditional form:

```text
if ( <classical-register> == <integer> ) <quantum-operation> ;
```

The condition always compares a **whole** classical register against an integer —
there is no bit-level or inequality form in the 2.0 grammar. That is why the
contract has a single `condition_register` string and a single `condition_value`
integer, and why those two columns are enough.

In [7]:
QREG = re.compile(r"^qreg\s+(\w+)\s*\[\s*(\d+)\s*\]$")
CREG = re.compile(r"^creg\s+(\w+)\s*\[\s*(\d+)\s*\]$")
CONDITIONAL = re.compile(r"^if\s*\(\s*(\w+)\s*==\s*(\d+)\s*\)\s*(.+)$")
OPERATION = re.compile(r"^([A-Za-z_]\w*)(\s*\([^)]*\))?\s+(.+)$")
MEASURE = re.compile(r"^measure\s+(\S+)\s*->\s*(\w+)(?:\[\s*\d+\s*\])?$")
INDEXED_QUBIT = re.compile(r"^(\w+)\s*\[\s*(\d+)\s*\]$")


def circuit_identity(member: str) -> tuple[str, str, str]:
    benchmark = member.split("/")[1]
    variant = "transpiled" if member.endswith("_transpiled.qasm") else "source"
    return f"qasmbench:{benchmark}:{variant}", benchmark, variant


def parse_circuit(member: str) -> dict:
    parsed = statements(archive.read(member).decode())
    circuit_id, benchmark, variant = circuit_identity(member)

    qregs, cregs, measured_at = {}, {}, {}
    for index, _line, text in parsed:
        if (match := QREG.match(text)) is not None:
            qregs[match.group(1)] = int(match.group(2))
        elif (match := CREG.match(text)) is not None:
            cregs[match.group(1)] = int(match.group(2))
        elif (match := MEASURE.match(text)) is not None:
            measured_at.setdefault(match.group(2), index)

    return {
        "member": member,
        "circuit_id": circuit_id,
        "benchmark_name": benchmark,
        "variant": variant,
        "statements": parsed,
        "qregs": qregs,
        "cregs": cregs,
        "measured_at": measured_at,
    }


CIRCUITS = {member: parse_circuit(member) for member in CIRCUIT_MEMBERS}

pd.DataFrame(
    [
        {
            "circuit_id": c["circuit_id"],
            "qregs": c["qregs"],
            "cregs": c["cregs"],
            "qubit_count": sum(c["qregs"].values()),
            "executed_statements": len(c["statements"]),
        }
        for c in CIRCUITS.values()
    ]
)

,circuit_id,qregs,cregs,qubit_count,executed_statements
0,qasmbench:error_correctiond3_n5:source,{'q': 5},{'c': 5},5,123
1,qasmbench:error_correctiond3_n5:transpiled,{'q': 5},{'c': 5},5,246
2,qasmbench:qec_en_n5:source,{'q': 5},{'c': 5},5,34
3,qasmbench:qec_en_n5:transpiled,{'q': 5},{'c': 5},5,55
4,qasmbench:qec_sm_n5:source,"{'q': 3, 'a': 2}","{'c': 3, 'syn': 2}",5,14
5,qasmbench:qec_sm_n5:transpiled,"{'q': 3, 'a': 2}","{'c': 3, 'syn': 2}",5,20


### `source_record_id`

The shared rule: *"a stable string that identifies the source record from which a
Silver row was produced. It must not depend on the run time or row order of a
database load."*

I use:

```text
qasmbench:<archive_member>#stmt=<statement_index>
```

Stable across runs because it is a pure function of the Bronze bytes. It
disambiguates two textually identical `if` lines in one file, which a
content hash would not.

**The tradeoff, stated openly:** the index is defined by *this* tokenizer. If
someone later changes what counts as a statement — say, keeps gate-definition
bodies — every id shifts. So the tokenizer definition is frozen together with the
id, and `source_trace.parquet` also carries the **line number**, which is a
property of the file rather than of the parser, as a human-checkable
cross-reference. Alternative considered and rejected: keying on the line number
alone — two statements can share a line (`cx d1,a1; cx d2,a1;`), so it is not
unique.

In [8]:
CONDITION_RULES = {
    "qasm_cc_condition_register_declared": "condition register must be a declared creg",
    "qasm_cc_condition_value_in_range": "condition value must fit the register width",
    "qasm_cc_single_target": "contract stores exactly one target_qubit",
    "qasm_cc_target_is_indexed_qubit": "target must be one indexed qubit, not a whole register",
    "qasm_cc_target_register_declared": "target register must be a declared qreg",
    "qasm_cc_target_index_in_range": "target index must be inside the declared register",
    "qasm_cc_condition_register_measured": "condition register should be written by a measure",
    "qasm_cc_condition_before_measure": "condition should appear after the measure that writes it",
}

candidates: list[dict] = []
issues: list[dict] = []


def record_issue(source_record_id, rule_id, severity, observed_value, action):
    issues.append(
        {
            "source_record_id": source_record_id,
            "rule_id": rule_id,
            "severity": severity,
            "observed_value": str(observed_value),
            "action": action,
            "reason": CONDITION_RULES[rule_id],
        }
    )


for circuit in CIRCUITS.values():
    for index, line_no, text in circuit["statements"]:
        match = CONDITIONAL.match(text)
        if match is None:
            continue

        register, value, operation = match.group(1), int(match.group(2)), match.group(3).strip()
        source_record_id = f"{SOURCE_NAME}:{circuit['member']}#stmt={index}"
        op_match = OPERATION.match(operation)
        gate = op_match.group(1) if op_match else None
        targets = [t.strip() for t in op_match.group(3).split(",")] if op_match else []

        accepted = True

        # --- condition side ---
        if register not in circuit["cregs"]:
            record_issue(source_record_id, "qasm_cc_condition_register_declared",
                         "error", register, "rejected")
            accepted = False
        elif not 0 <= value < 2 ** circuit["cregs"][register]:
            record_issue(source_record_id, "qasm_cc_condition_value_in_range",
                         "error", value, "rejected")
            accepted = False

        # --- target side ---
        if len(targets) != 1:
            record_issue(source_record_id, "qasm_cc_single_target", "error", operation, "rejected")
            accepted = False
        else:
            target_match = INDEXED_QUBIT.match(targets[0])
            if target_match is None:
                record_issue(source_record_id, "qasm_cc_target_is_indexed_qubit",
                             "error", targets[0], "rejected")
                accepted = False
            elif target_match.group(1) not in circuit["qregs"]:
                record_issue(source_record_id, "qasm_cc_target_register_declared",
                             "error", targets[0], "rejected")
                accepted = False
            elif int(target_match.group(2)) >= circuit["qregs"][target_match.group(1)]:
                record_issue(source_record_id, "qasm_cc_target_index_in_range",
                             "error", targets[0], "rejected")
                accepted = False

        # --- semantic warnings: a correction should react to a measured syndrome ---
        measured = circuit["measured_at"].get(register)
        if register in circuit["cregs"] and measured is None:
            record_issue(source_record_id, "qasm_cc_condition_register_measured",
                         "warning", register, "accepted")
        elif measured is not None and measured > index:
            record_issue(source_record_id, "qasm_cc_condition_before_measure",
                         "warning", f"measure at stmt={measured}, condition at stmt={index}",
                         "accepted")

        candidates.append(
            {
                "source_record_id": source_record_id,
                "circuit_id": circuit["circuit_id"],
                "condition_register": register,
                "condition_value": value,
                "gate": gate,
                "target_qubit": targets[0] if len(targets) == 1 else None,
                "archive_member": circuit["member"],
                "line_number": line_no,
                "statement_index": index,
                "statement_text": text,
                "accepted": accepted,
            }
        )

candidate_frame = pd.DataFrame(candidates)
# archive_member / line_number / statement_index / statement_text / accepted are
# discovery-only working columns. Only the six contract columns reach Silver (cell below).
candidate_frame[
    ["source_record_id", "circuit_id", "condition_register", "condition_value",
     "gate", "target_qubit", "line_number", "accepted"]
]

,source_record_id,circuit_id,condition_register,condition_value,gate,target_qubit,line_number,accepted
0,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=10,qasmbench:qec_sm_n5:source,syn,1,x,q[0],17,True
1,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=11,qasmbench:qec_sm_n5:source,syn,2,x,q[2],18,True
2,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=12,qasmbench:qec_sm_n5:source,syn,3,x,q[1],19,True
3,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=14,qasmbench:qec_sm_n5:transpiled,syn,1,x,q[0],15,True
4,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=16,qasmbench:qec_sm_n5:transpiled,syn,2,x,q[2],17,True
5,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=17,qasmbench:qec_sm_n5:transpiled,syn,3,x,q[1],18,True


## 8. Reconciliation and issues

`rubric.md` wants *"accepted and rejected counts reconcile"*, and the crude text
scan from §4 gives an independent number to reconcile against.

In [9]:
read = len(candidate_frame)
accepted = int(candidate_frame["accepted"].sum())
rejected = read - accepted

print(f"crude `if(` text scan   : {CRUDE_CONDITION_COUNT}")
print(f"statements parsed       : {read}")
print(f"accepted into Silver    : {accepted}")
print(f"rejected -> data_issues : {rejected}")
assert read == CRUDE_CONDITION_COUNT, "parser and text scan disagree"

issue_frame = pd.DataFrame(issues, columns=["source_record_id", "rule_id", "severity",
                                            "observed_value", "action", "reason"])
print(f"\nissue rows: {len(issue_frame)}")
issue_frame

crude `if(` text scan   : 6
statements parsed       : 6
accepted into Silver    : 6
rejected -> data_issues : 0

issue rows: 0


,source_record_id,rule_id,severity,observed_value,action,reason


Every rule passes on this release, so `data_issues.parquet` gets nothing from this
table. That is a result, not an excuse to drop the checks: the rubric grades
*tested rules*, and a rule that never fires still has to be unit-tested against a
crafted bad input. Candidates for those tests:

| Rule | Synthetic input that should fail it |
| --- | --- |
| `condition_value_in_range` | `creg syn[2];` + `if(syn==4) x q[0];` |
| `single_target` | `if(syn==1) cx q[0],q[1];` |
| `target_is_indexed_qubit` | `if(syn==1) x q;` (whole register) |
| `target_register_declared` | `if(syn==1) x z[0];` |
| `target_index_in_range` | `qreg q[3];` + `if(syn==1) x q[7];` |
| `condition_register_declared` | `if(nope==1) x q[0];` |
| `condition_before_measure` | move the `if` above `measure a -> syn` |

Those belong in `starter/tests/`, built from tiny inline QASM strings — no Bronze
read needed.

## 9. Cross-check: does the correction table agree with the parity structure?

This is the strongest check available, and it is *not* a schema check. Silver
table 5 (`stabilizer_check`) records which data qubits each ancilla covers and
which syndrome bit it writes. From that structure alone I can predict what the
corrections should be, then compare against what the file actually says.

For the transpiled variant the structure is explicit:

```text
cx q[0],a[0];  cx q[1],a[0];   ->  ancilla a[0] checks parity of {q0, q1}
cx q[1],a[1];  cx q[2],a[1];   ->  ancilla a[1] checks parity of {q1, q2}
measure a[0] -> syn[0];        ->  a[0]'s outcome is syndrome bit 0 (value 1)
measure a[1] -> syn[1];        ->  a[1]'s outcome is syndrome bit 1 (value 2)
```

Decoding rule for a repetition code: the fired checks are the ones whose parity
broke. A single X error on a data qubit breaks exactly the checks that contain it.
So invert it — given a set of fired checks, the culprit is the qubit contained in
every fired check and in no unfired check.

In [10]:
CX = re.compile(r"^cx\s+(\w+\[\d+\])\s*,\s*(\w+\[\d+\])$")
MEASURE_BIT = re.compile(r"^measure\s+(\w+\[\d+\])\s*->\s*(\w+)\[\s*(\d+)\s*\]$")

transpiled = CIRCUITS["small/qec_sm_n5/qec_sm_n5_transpiled.qasm"]
corrections_here = candidate_frame[candidate_frame.circuit_id == transpiled["circuit_id"]]
condition_registers = set(corrections_here["condition_register"])

# A cx writes parity of its control into its target; collect target -> {controls}.
covers: dict[str, set[str]] = {}
measured_into: dict[str, tuple[str, int]] = {}
for _index, _line, text in transpiled["statements"]:
    if (match := CX.match(text)) is not None:
        covers.setdefault(match.group(2), set()).add(match.group(1))
    elif (match := MEASURE_BIT.match(text)) is not None:
        measured_into[match.group(1)] = (match.group(2), int(match.group(3)))

# A qubit is an ancilla only if it accumulates parity AND is measured into a
# register that some correction actually reads. Without the second condition
# `measure q[0] -> c[0]` would look like a syndrome bit too.
syndrome_bit = {
    qubit: (register, bit)
    for qubit, (register, bit) in measured_into.items()
    if qubit in covers and register in condition_registers
}
covers = {ancilla: data for ancilla, data in covers.items() if ancilla in syndrome_bit}

print("parity checks read out of the circuit:")
for ancilla in sorted(covers):
    register, bit = syndrome_bit[ancilla]
    print(f"  {ancilla} checks {sorted(covers[ancilla])} -> {register}[{bit}] (value {2 ** bit})")

data_qubits = sorted({q for group in covers.values() for q in group})


def decode(value: int) -> tuple[list[str], str | None]:
    """Minimum-weight decode: the qubit inside every fired check and no other."""
    fired = {a for a, (_register, bit) in syndrome_bit.items() if value >> bit & 1}
    culprits = [
        qubit
        for qubit in data_qubits
        if {a for a in covers if qubit in covers[a]} == fired
    ]
    return sorted(fired), culprits[0] if len(culprits) == 1 else None


actual = dict(zip(corrections_here["condition_value"], corrections_here["target_qubit"]))
comparison = pd.DataFrame(
    [
        {
            "condition_value": value,
            "fired_checks": fired,
            "predicted_target": predicted,
            "actual_target": actual.get(value),
            "agrees": predicted == actual.get(value),
        }
        for value in range(1, 2 ** len(covers))
        for fired, predicted in [decode(value)]
    ]
)
assert comparison["agrees"].all(), "correction table disagrees with the parity structure"
comparison
comparison

parity checks read out of the circuit:
  a[0] checks ['q[0]', 'q[1]'] -> syn[0] (value 1)
  a[1] checks ['q[1]', 'q[2]'] -> syn[1] (value 2)


,condition_value,fired_checks,predicted_target,actual_target,agrees
0,1,[a[0]],q[0],q[0],True
1,2,[a[1]],q[2],q[2],True
2,3,"[a[0], a[1]]",q[1],q[1],True


All three agree. Two things follow.

1. **My six rows are semantically right**, not just syntactically parsed. `syn==3`
   (both checks fired) points at `q[1]`, the qubit shared by both checks — exactly
   the minimum-weight decode.
2. **This is the answer to Part I analysis question 3** — *"How does the
   repetition-code circuit map data qubits to parity-check ancillas, syndrome
   bits, and conditional corrections?"* Once tables 4/5/6 are in Gold that becomes
   a three-table SQL join, which also satisfies the "at least one query joins
   three or more Gold tables" requirement. Worth telling whoever owns the Gold
   analyses that this question is served by my table.

Note `syn==0` has no correction, and that is correct: no check fired, nothing to
undo. Do not record a missing row for it.

## 10. Contract dry-run

Build the exact six columns with the exact declared types and confirm the frame
matches. **No write** — the Silver stage does that.

In [11]:
CONDITIONAL_CORRECTION_SCHEMA = pa.schema(
    [
        ("source_record_id", pa.string()),
        ("circuit_id", pa.string()),
        ("condition_register", pa.string()),
        ("condition_value", pa.int64()),
        ("gate", pa.string()),
        ("target_qubit", pa.string()),
    ]
)

silver = (
    candidate_frame.loc[candidate_frame["accepted"], [f.name for f in CONDITIONAL_CORRECTION_SCHEMA]]
    .sort_values(["circuit_id", "condition_value"])
    .reset_index(drop=True)
)
table = pa.Table.from_pandas(silver, schema=CONDITIONAL_CORRECTION_SCHEMA, preserve_index=False)

assert table.schema.equals(CONDITIONAL_CORRECTION_SCHEMA)
assert silver["source_record_id"].is_unique
assert not silver.isna().any().any()
assert not silver.duplicated(["circuit_id", "condition_register", "condition_value"]).any()
assert set(silver["circuit_id"]) <= {c["circuit_id"] for c in CIRCUITS.values()}

print(table.schema)
print(f"\nrows: {table.num_rows}")
silver

source_record_id: string
circuit_id: string
condition_register: string
condition_value: int64
gate: string
target_qubit: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 822

rows: 6


,source_record_id,circuit_id,condition_register,condition_value,gate,target_qubit
0,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=10,qasmbench:qec_sm_n5:source,syn,1,x,q[0]
1,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=11,qasmbench:qec_sm_n5:source,syn,2,x,q[2]
2,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=12,qasmbench:qec_sm_n5:source,syn,3,x,q[1]
3,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=14,qasmbench:qec_sm_n5:transpiled,syn,1,x,q[0]
4,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=16,qasmbench:qec_sm_n5:transpiled,syn,2,x,q[2]
5,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=17,qasmbench:qec_sm_n5:transpiled,syn,3,x,q[1]


The last two assertions are the ones worth arguing about.

`(circuit_id, condition_register, condition_value)` unique says *one syndrome value
triggers at most one correction in a circuit*. True here. It is **not** a law of
OpenQASM — a circuit could legally apply two corrections for one value
(`if(syn==1) x q[0]; if(syn==1) z q[0];`). So this is a **contract check on this
release**, not a primary key. If it ever fires, the answer is to record the issue,
not to drop a row: the grain is "one recovery operation", so two operations are two
rows. The real key is `source_record_id`.

`circuit_id ⊆ table 4's circuit ids` is a foreign key I cannot enforce alone —
see below.

## 11. What `source_trace.parquet` needs from me

Tracing lives in `results/part1/`, not in the Silver table. My rows contribute
one trace row each — unlike a Google shot, which needs several because it is
assembled from aligned companion files.

In [12]:
trace = pd.DataFrame(
    [
        {
            "source_record_id": row.source_record_id,
            "source_name": SOURCE_NAME,
            "bronze_object": BRONZE_OBJECT,
            "archive_member": row.archive_member,
            "record_locator": f"statement_index={row.statement_index};line={row.line_number}",
            "input_sha256": INPUT_SHA256,
        }
        for row in candidate_frame.itertuples()
        if row.accepted
    ]
)
trace

,source_record_id,source_name,bronze_object,archive_member,record_locator,input_sha256
0,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=10,qasmbench,bronze/source=qasmbench/qasmbench-qec.zip,small/qec_sm_n5/qec_sm_n5.qasm,statement_index=10;line=17,60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e...
1,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=11,qasmbench,bronze/source=qasmbench/qasmbench-qec.zip,small/qec_sm_n5/qec_sm_n5.qasm,statement_index=11;line=18,60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e...
2,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=12,qasmbench,bronze/source=qasmbench/qasmbench-qec.zip,small/qec_sm_n5/qec_sm_n5.qasm,statement_index=12;line=19,60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e...
3,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=14,qasmbench,bronze/source=qasmbench/qasmbench-qec.zip,small/qec_sm_n5/qec_sm_n5_transpiled.qasm,statement_index=14;line=15,60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e...
4,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=16,qasmbench,bronze/source=qasmbench/qasmbench-qec.zip,small/qec_sm_n5/qec_sm_n5_transpiled.qasm,statement_index=16;line=17,60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e...
5,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=17,qasmbench,bronze/source=qasmbench/qasmbench-qec.zip,small/qec_sm_n5/qec_sm_n5_transpiled.qasm,statement_index=17;line=18,60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e...


## 12. Repeatability

*"A second run on unchanged input must produce the same records and identifiers."*
Re-parse from the same bytes and compare. This is the notebook version of the test
that belongs in `starter/tests/`.

In [13]:
def build_corrections(raw: bytes) -> pd.DataFrame:
    """Re-derive the accepted rows from raw Bronze bytes only."""
    zf = zipfile.ZipFile(io.BytesIO(raw))
    rows = []
    for member in sorted(n for n in zf.namelist() if n.endswith(".qasm")):
        circuit_id, _benchmark, _variant = circuit_identity(member)
        for index, _line, text in statements(zf.read(member).decode()):
            match = CONDITIONAL.match(text)
            if match is None:
                continue
            op_match = OPERATION.match(match.group(3).strip())
            targets = [t.strip() for t in op_match.group(3).split(",")]
            rows.append(
                {
                    "source_record_id": f"{SOURCE_NAME}:{member}#stmt={index}",
                    "circuit_id": circuit_id,
                    "condition_register": match.group(1),
                    "condition_value": int(match.group(2)),
                    "gate": op_match.group(1),
                    "target_qubit": targets[0],
                }
            )
    return (
        pd.DataFrame(rows, columns=[f.name for f in CONDITIONAL_CORRECTION_SCHEMA])
        .sort_values(["circuit_id", "condition_value"])
        .reset_index(drop=True)
    )


first = build_corrections(BRONZE_BYTES)
second = build_corrections(fetch(BRONZE_OBJECT))

pd.testing.assert_frame_equal(first, second)
pd.testing.assert_frame_equal(first, silver)
print("stable across re-reads, and identical to the frame built above")
print(f"ids: {sorted(first.source_record_id)}")

stable across re-reads, and identical to the frame built above
ids: ['qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=10', 'qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=11', 'qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=12', 'qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=14', 'qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=16', 'qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=17']


## 13. Findings and open questions

### Findings (verified above against this release)

| # | Finding | Consequence |
| --- | --- | --- |
| 1 | `input_sha256` of the Bronze object recorded in cell 0 | goes into `run.json` and every `source_trace` row |
| 2 | No unsafe archive member names | run may proceed; the check must stay in the stage |
| 3 | 19 members, 6 of them `.qasm`; `qelib1.inc` is a gate library, not a circuit | table 4 has 6 rows, not 3 and not 19 |
| 4 | Only `qec_sm_n5` has conditional corrections; 3 per variant | **table 6 has exactly 6 rows**; the other four circuits legitimately contribute none |
| 5 | Every condition is `if(<creg> == <int>)` on a whole register | the two condition columns are sufficient; no bit-level form exists in OpenQASM 2.0 |
| 6 | Every target is a single indexed qubit, gate `x` | one `target_qubit` column is sufficient *for this release* |
| 7 | `syn` is 2 bits; values 1, 2, 3 present, 0 absent | correct — value 0 means no check fired, so no correction |
| 8 | Corrections match the parity structure independently derived from the `cx`/`measure` statements | table 6 is semantically consistent with table 5; answers analysis question 3 |
| 9 | Zero rows rejected, zero issues | `data_issues.parquet` gets nothing from this table; rules still need unit tests |
| 10 | Source and transpiled variants carry identical corrections at different statement positions | ids must be per-variant; do not deduplicate across variants |

### Open questions for the team

1. **`circuit_id` format.** I used `qasmbench:<benchmark>:<variant>`. Whoever owns
   Silver table 4 owns this string — my `circuit_id` column is a foreign key into
   theirs. **This has to be agreed before either table is written.** If they use
   the archive member path instead, I change; it costs one line.
2. **Does `stabilizer_check` (table 5) cover the `source` variant?** Its checks are
   inside the `gate syndrome` definition body, so they need macro expansion.
   `data-sources.md` says "where explicit", which leaves room to populate table 5
   from the transpiled variant only. My table 6 covers both variants either way,
   so a decision there does not block me — but it makes the Gold join asymmetric
   and someone should decide deliberately.
3. **Gold grain.** In Gold, is a conditional correction its own entity, or is it an
   attribute of a stabilizer-check outcome? Not my call at Silver scope, but table 6
   plus table 5 is what makes analysis question 3 a three-table join, so the Gold
   owner should know it is available.

### What is explicitly out of scope here

- Writing anything to MinIO or PostgreSQL — the stage script under
  `src/quantum_lake_student/stages/` does that.
- Simulating the circuits. `data-sources.md`: *"Do not simulate the circuits."*
- Turning QASMBench into training rows. `faq.md`: it has no shot labels. It is
  structural context for Gold and nothing else.